# NB-R05 — Walk-Forward Generalizability Check

**Pipeline stage:** 5 of 13

**Purpose.** Test whether the High-VIX regime effect observed in the primary test window (2025-2026) generalizes across other years, or whether it is specific to the single stress episode contained in that window.

**Why this matters.** Every High-VIX observation in the primary test set falls within one concentrated volatility episode. A single episode cannot distinguish a genuine, repeatable regime effect from an artifact of that particular episode's other characteristics. This notebook runs an annual walk-forward evaluation: for each year *Y*, a model is trained on all data up to *Y* and evaluated on year *Y*+1, with the regime threshold recalibrated from the corresponding pre-test window at each step.

**Inputs:** `data/raw/market_data.csv`, `data/raw/india_vix.csv`.

**Outputs:** `results/walkforward_results.csv`, `plots/R05_walkforward_regime_accuracy.png`.

**Result:** across six independently evaluated years (2019, 2020, 2021, 2022, 2024, 2025), the High-VIX accuracy advantage holds in only one (2021, +12.4 pp) and reverses in the other five (-11.6 to -39.3 pp). This is the single most important robustness result in the study: it indicates the primary-sample finding is concentrated in, and likely specific to, the one evaluated stress episode rather than reflecting a general, repeatable regime effect.


In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier

PROJ    = Path('..').resolve()  # repo root, assuming this notebook is run from notebooks/
PROC    = PROJ / 'data' / 'processed'
RAW     = PROJ / 'data' / 'raw'
RESULTS = PROJ / 'results'
PLOTS   = PROJ / 'plots'
MODELS  = PROJ / 'models'

with open(PROC / 'feature_cols.json') as f:
    FEATURE_COLS = json.load(f)
with open(RESULTS / 'vix_threshold_config.json') as f:
    vix_cfg = json.load(f)
with open(RESULTS / 'all_best_params.json') as f:
    best_params = json.load(f)

SEED = 42
HORIZON = 21
MIN_TRAIN_ROWS = 500   # minimum training rows for a walk-forward window
VIX_THRESHOLD = vix_cfg['vix_threshold_fixed']

print(f'VIX threshold: {VIX_THRESHOLD:.2f}')

## 1. Reload Full Dataset and Engineer Features

In [ ]:
# Load from cleaned splits and reconstruct full frame
train = pd.read_csv(PROC / 'train.csv', parse_dates=['date'])
val   = pd.read_csv(PROC / 'val.csv',   parse_dates=['date'])
test  = pd.read_csv(PROC / 'test_with_regimes.csv', parse_dates=['date'])

# Re-read raw (unscaled) data for walk-forward (we need to refit scaler per window)
bn  = pd.read_csv(RAW / 'market_data.csv', parse_dates=['date'])
vix = pd.read_csv(RAW / 'india_vix.csv',   parse_dates=['date'])
bn.columns  = [c.strip().lower().replace(' ','_') for c in bn.columns]
vix.columns = [c.strip().lower().replace(' ','_') for c in vix.columns]

bn_date_col  = [c for c in bn.columns  if 'date' in c][0]
vix_date_col = [c for c in vix.columns if 'date' in c][0]
bn  = bn.rename(columns={bn_date_col:  'date'})
vix = vix.rename(columns={vix_date_col:'date'})

vix_close_col = [c for c in vix.columns if 'close' in c or 'vix' in c.lower()][0]
vix_slim = vix[['date', vix_close_col]].rename(columns={vix_close_col:'india_vix'})

df = bn.merge(vix_slim, on='date', how='inner').sort_values('date').reset_index(drop=True)
print(f'Full dataset: {len(df)} rows | {df.date.min().date()} -> {df.date.max().date()}')

In [ ]:
# Feature engineering (same as NB-R01)
close_col = [c for c in df.columns if 'close' in c and 'india' not in c][0]
high_col  = [c for c in df.columns if 'high' in c][0]
low_col   = [c for c in df.columns if 'low' in c][0]
close = df[close_col]; high = df[high_col]; low = df[low_col]

ema12 = close.ewm(span=12, adjust=False).mean()
ema26 = close.ewm(span=26, adjust=False).mean()
df['macd']        = ema12 - ema26
df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
df['macd_hist']   = df['macd'] - df['macd_signal']
df['ema20']       = close.ewm(span=20, adjust=False).mean()

delta = close.diff(); gain = delta.clip(lower=0); loss = (-delta).clip(lower=0)
df['rsi14'] = 100 - (100 / (1 + gain.ewm(com=13,adjust=False).mean() / loss.ewm(com=13,adjust=False).mean().replace(0,np.nan)))

low14 = low.rolling(14).min(); high14 = high.rolling(14).max()
df['stoch_k'] = 100*(close-low14)/(high14-low14).replace(0,np.nan)
df['stoch_d'] = df['stoch_k'].rolling(3).mean()
df['roc10']   = close.pct_change(10)*100

bb_mid = close.rolling(20).mean(); bb_std = close.rolling(20).std()
df['bb_upper'] = bb_mid + 2*bb_std; df['bb_lower'] = bb_mid - 2*bb_std
df['bb_width'] = (df['bb_upper']-df['bb_lower'])/bb_mid

tr = pd.concat([high-low,(high-close.shift(1)).abs(),(low-close.shift(1)).abs()],axis=1).max(axis=1)
df['atr14'] = tr.ewm(com=13,adjust=False).mean()

for k in [1,2,3,5]:
    df[f'log_ret_lag{k}'] = np.log(close/close.shift(k))

df['close_fwd21'] = close.shift(-HORIZON)
df['dir_21d'] = (df['close_fwd21'] > close).astype(float)
df.loc[df['close_fwd21'].isna(), 'dir_21d'] = np.nan

df_clean = df.dropna(subset=FEATURE_COLS+['dir_21d','india_vix']).copy()
print(f'Clean dataset: {len(df_clean)} rows')

## 2. Identify Historical High-VIX Episodes

In [ ]:
# Assign regime using the pre-specified threshold (no look-ahead)
df_clean['regime'] = np.where(df_clean['india_vix'] >= VIX_THRESHOLD, 'High-VIX', 'Low-VIX')

# Identify distinct High-VIX episodes (strictly contiguous stretches)
is_high = df_clean['regime'].eq('High-VIX')
run_id = is_high.ne(is_high.shift(fill_value=False)).cumsum()
episodes = (
    df_clean[is_high]
    .groupby(run_id[is_high])
    .agg(start=('date', 'min'), end=('date', 'max'), days=('date', 'size'))
    .reset_index(drop=True)
)

print(f'Number of distinct High-VIX episodes: {len(episodes)}')
for row in episodes.itertuples(index=False):
    print(f'  Episode start: {row.start.date()} | end: {row.end.date()} | days: {row.days}')

## 3. Walk-Forward Regime Accuracy Test

In [ ]:
walk_results = []

# Define annual evaluation windows (each year's data as test, prior as train)
years = sorted(df_clean['date'].dt.year.unique())

for test_year in years:
    test_mask  = df_clean['date'].dt.year == test_year
    train_mask = df_clean['date'].dt.year < test_year

    test_yr  = df_clean[test_mask]
    train_yr = df_clean[train_mask]

    if len(train_yr) < MIN_TRAIN_ROWS:
        continue

    n_high = (test_yr['regime'] == 'High-VIX').sum()
    n_low  = (test_yr['regime'] == 'Low-VIX').sum()
    if n_high < 5 or n_low < 5:
        continue

    # Scale features
    sc = StandardScaler()
    X_tr = sc.fit_transform(train_yr[FEATURE_COLS].values)
    X_te = sc.transform(test_yr[FEATURE_COLS].values)
    y_tr = train_yr['dir_21d'].values
    y_te = test_yr['dir_21d'].values

    # Quick XGBoost (fixed params for speed)
    m = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                           use_label_encoder=False, eval_metric='logloss',
                           random_state=SEED, n_jobs=-1, verbosity=0)
    m.fit(X_tr, y_tr)
    probs = m.predict_proba(X_te)[:, 1]
    preds = (probs >= 0.5).astype(int)

    reg = test_yr['regime'].values
    high_m = reg == 'High-VIX'
    low_m  = reg == 'Low-VIX'

    acc_high = accuracy_score(y_te[high_m], preds[high_m]) * 100
    acc_low  = accuracy_score(y_te[low_m],  preds[low_m])  * 100
    diff     = acc_high - acc_low

    walk_results.append({
        'year': test_year,
        'n_train': len(train_yr), 'n_test': len(test_yr),
        'n_high_vix': int(n_high), 'n_low_vix': int(n_low),
        'acc_high_vix': acc_high, 'acc_low_vix': acc_low,
        'diff_pp': diff, 'high_gt_low': diff > 0
    })
    print(f'{test_year}: High={acc_high:.1f}%, Low={acc_low:.1f}%, Diff={diff:+.1f}pp (n_high={n_high})')

wf_df = pd.DataFrame(walk_results)
wf_df.to_csv(RESULTS / 'walkforward_results.csv', index=False)

n_consistent = wf_df['high_gt_low'].sum()
print(f'\nHigh-VIX > Low-VIX in {n_consistent}/{len(wf_df)} years ({n_consistent/len(wf_df)*100:.0f}%)')

In [ ]:
# Plot walk-forward results
fig, ax = plt.subplots(figsize=(10, 5))
x = wf_df['year']
ax.bar(x - 0.2, wf_df['acc_high_vix'], width=0.35, label='High-VIX accuracy', color='tomato', alpha=0.8)
ax.bar(x + 0.2, wf_df['acc_low_vix'],  width=0.35, label='Low-VIX accuracy',  color='steelblue', alpha=0.8)
ax.axhline(50, color='black', linestyle='--', linewidth=1, label='Random baseline (50%)')
ax.set_xlabel('Test Year')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Walk-Forward Regime Accuracy by Year (XGBoost, 21d Horizon)')
ax.set_xticks(x)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(PLOTS / 'R05_walkforward_regime_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

---
## Summary

**Pipeline stage:** 5 of 13 (see `notebooks/README.md` for the full pipeline map).

**Artifacts produced by this notebook:**

- `results/walkforward_results.csv`
- `plots/R05_walkforward_regime_accuracy.png`

**Next notebook:** `NB-R06_baseline_comparison.ipynb`
